In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
import yaml
from pyspark.sql import functions as F

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
from riesgo_credito.procesamiento import esquema, etl

DATOS = RAIZ / "data"
TRIMESTRE = "2019Q1"
PARTICION = f"trimestre={TRIMESTRE}"
RUTAS = {
    "originacion": DATOS / "limpios" / "originacion" / PARTICION,
    "desempeno": DATOS / "limpios" / "desempeno" / PARTICION,
    "p1": DATOS / "modelado" / "p1" / PARTICION,
    "p2": DATOS / "modelado" / "p2" / PARTICION,
    "tablero_originacion": DATOS / "tablero" / "originacion" / PARTICION,
    "tablero_serie_mensual": DATOS / "tablero" / "serie_mensual" / PARTICION,
    "tablero_cartera_activa": DATOS / "tablero" / "cartera_activa" / PARTICION,
}

params = yaml.safe_load((RAIZ / "params.yaml").read_text())
spark = etl.crear_sesion(params["spark"]["memoria_driver"], params["spark"]["particiones"])
spark.sparkContext.setLogLevel("ERROR")
leer = {n: spark.read.parquet(str(r)) for n, r in RUTAS.items()}

metricas = pd.DataFrame([json.loads(p.read_text()) for p in sorted((DATOS / "metricas" / "etl").glob("*.json"))]).set_index("trimestre")
perfil = pd.DataFrame([{"trimestre": p.stem, "creditos_crudos": (d := json.loads(p.read_text()))["originacion"]["filas"],
                        "registros_crudos": d["desempeno"]["filas"],
                        "valuacion_sin_dato_pct": 100 * d["originacion"]["perfil"]["valuation_method"]["no_disponible"]["7"]
                        / d["originacion"]["filas"]}
                       for p in sorted((DATOS / "metricas" / "perfil_crudos").glob("*.json"))]).set_index("trimestre")
orig = pd.read_parquet(RUTAS["originacion"])
p1 = pd.read_parquet(RUTAS["p1"])
print(f"Trimestres procesados: {len(metricas)} de {len(perfil)} medidos en los crudos")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/31 19:08:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Trimestres procesados: 29 de 29 medidos en los crudos


## Todos los trimestres

In [2]:
resumen = metricas[["creditos", "registros_desempeno"]].join(perfil)
resumen["coincide"] = (resumen.creditos == resumen.creditos_crudos) & (resumen.registros_desempeno == resumen.registros_crudos)
resumen["p1_cobertura_pct"] = 100 * metricas.p1_etiquetables / metricas.p1_creditos
resumen["p1_tasa_pct"] = 100 * metricas.p1_positivos / metricas.p1_etiquetables
resumen["p2_tasa_pct"] = 100 * metricas.p2_positivos / metricas.p2_etiquetables
assert resumen.coincide.all()
resumen[["creditos", "registros_desempeno", "coincide", "p1_cobertura_pct", "p1_tasa_pct", "p2_tasa_pct"]].round(3)

,creditos,registros_desempeno,coincide,p1_cobertura_pct,p1_tasa_pct,p2_tasa_pct
trimestre,,,,,,
2019Q1,279861,9821276,True,100.000,0.244,0.167
2019Q2,413821,15089232,True,100.000,1.765,0.164
2019Q3,542237,22134947,True,100.000,3.324,0.139
2019Q4,545225,22584658,True,100.000,3.757,0.137
2020Q1,517072,22186576,True,100.000,2.957,0.090
2020Q2,926253,46117163,True,100.000,0.842,0.035
2020Q3,1187782,64417206,True,100.000,0.464,0.026
2020Q4,1282634,70364472,True,100.000,0.367,0.024
2021Q1,1219583,65420876,True,100.000,0.310,0.024


In [3]:
metricas[["excluidos_codigo_96", "registros_xx", "registros_xx_despues_primer_pago", "tasas_mayores_20",
          "tasas_mayores_doble_original", "tasa_actual_maxima"]].join(perfil.valuacion_sin_dato_pct.round(1))

,excluidos_codigo_96,registros_xx,registros_xx_despues_primer_pago,tasas_mayores_20,tasas_mayores_doble_original,tasa_actual_maxima,valuacion_sin_dato_pct
trimestre,,,,,,,
2019Q1,801,0,0,0,0,7.000,94.3
2019Q2,1078,0,0,0,0,6.750,91.3
2019Q3,1184,0,0,0,0,6.750,86.9
2019Q4,1268,0,0,0,0,6.750,84.4
2020Q1,2027,0,0,0,0,6.500,0.0
2020Q2,1851,0,0,0,0,7.125,0.0
2020Q3,2109,0,0,0,0,6.250,0.0
2020Q4,2265,0,0,0,0,6.500,0.0
2021Q1,2196,0,0,0,0,6.500,0.0


En los 29 trimestres el ETL lee exactamente los créditos y registros que cuenta el perfil de los archivos crudos, que se calculó por separado. La tasa de la pregunta 1 sube en dos grupos por razones distintas. En las cosechas de 2019Q2 a 2020Q1 el primer año coincidió con la pandemia y la indulgencia infla la etiqueta. En las más recientes la ventana de 12 meses aún no termina: la cobertura cae y solo tienen etiqueta los créditos que ya incumplieron o ya salieron, así que esas tasas no son comparables y no sirven para entrenar ni para la curva por cohorte del tablero. En la segunda tabla, los meses sin dato de mora están todos antes del primer pago, las tasas actuales fuera de rango son casos aislados que se conservan como vienen y el método de valoración falta casi siempre en 2019.

## Cifras de referencia de 2019Q1

In [4]:
eda = {"creditos": 279861, "registros_desempeno": 9821276, "excluidos_codigo_96": 801,
       "p1_etiquetables": 279060, "p1_positivos": 680,
       "p2_etiquetables": 9129854, "p2_positivos": 14938, "p2_30_etiquetables": 9236499, "p2_30_positivos": 35421,
       "p2_60_etiquetables": 9129930, "p2_60_positivos": 42799, "activos_al_corte": 55339}
# Desde el primer pago: el EDA incluía en P2 los meses previos al primer pago
referencia = {**eda, "p2_etiquetables": 8875387, "p2_positivos": 14863, "p2_30_etiquetables": 8982024,
              "p2_30_positivos": 35345, "p2_60_etiquetables": 8875463, "p2_60_positivos": 42411}
comparacion = pd.DataFrame({"eda": eda, "referencia": referencia, "etl": metricas.loc[TRIMESTRE, list(eda)]})
comparacion["coincide"] = comparacion.referencia == comparacion.etl
assert comparacion.coincide.all()
comparacion

,eda,referencia,etl,coincide
creditos,279861,279861,279861,True
registros_desempeno,9821276,9821276,9821276,True
excluidos_codigo_96,801,801,801,True
p1_etiquetables,279060,279060,279060,True
p1_positivos,680,680,680,True
p2_etiquetables,9129854,8875387,8875387,True
p2_positivos,14938,14863,14863,True
p2_30_etiquetables,9236499,8982024,8982024,True
p2_30_positivos,35421,35345,35345,True
p2_60_etiquetables,9129930,8875463,8875463,True


En la pregunta 1 y en los conteos generales el ETL reproduce exactamente el EDA. En la pregunta 2 la referencia es la regla del EDA más un filtro: solo meses desde el primer pago. Freddie Mac marca los meses previos al primer pago como al día en los trimestres antiguos y como XX en los recientes, así que dejarlos haría que la tabla cambiara de definición entre trimestres. El filtro quita el 2,8 % de las filas y 75 positivos, que son créditos que no pagaron desde la primera cuota y que ya cubre la pregunta 1.

## Calidad de las salidas de 2019Q1

In [5]:
perf, p2 = leer["desempeno"], leer["p2"]
sin_defecto = set(orig.loc[~orig.defecto, "loan_id"])
defecto = spark.createDataFrame(orig.loc[orig.defecto, ["loan_id"]])
chequeos = pd.Series({
    "duplicados en originación": orig.loan_id.duplicated().sum(),
    "duplicados en desempeño (crédito-mes)": perf.count() - perf.select("loan_id", "period").distinct().count(),
    "duplicados en P1": p1.loan_id.duplicated().sum(),
    "duplicados en P2 (crédito-mes)": p2.count() - p2.select("loan_id", "period").distinct().count(),
    "créditos sin defecto que faltan en P1": len(sin_defecto - set(p1.loan_id)),
    "créditos con código 96 en P1": len(set(p1.loan_id) - sin_defecto),
    "créditos con código 96 en P2": p2.join(defecto, "loan_id").select("loan_id").distinct().count(),
    "meses de P2 antes del primer pago": p2.where(F.col("edad") < 1).count(),
    "meses de P2 con atraso distinto de 0 o 30 días": p2.where(~F.col("dlq_num").isin(0, 1)).count(),
    "columnas posteriores al cierre en desempeño o P2": len(set(esquema.POSTERIORES_AL_CIERRE) & set(perf.columns + p2.columns)),
    "columnas fuera de la solicitud en P1": len(set(p1.columns) - set(etl.VARIABLES_P1) - {"loan_id", "first_payment_date", "y_p1", "trimestre"}),
    "puntajes 9999 en P1": (p1.credit_score == 9999).sum(),
    "deuda-ingreso 999 en P1": (p1.dti == 999).sum(),
    "registros previos al primer pago en desempeño (edad <= 0)": perf.where(F.col("edad") <= 0).count(),
}, name="valor")
chequeos.to_frame()

,valor
duplicados en originación,0
duplicados en desempeño (crédito-mes),0
duplicados en P1,0
duplicados en P2 (crédito-mes),0
créditos sin defecto que faltan en P1,0
créditos con código 96 en P1,0
créditos con código 96 en P2,0
meses de P2 antes del primer pago,0
meses de P2 con atraso distinto de 0 o 30 días,0
columnas posteriores al cierre en desempeño o P2,0


In [6]:
n_p2 = p2.count()
nulos_p2 = p2.select([(100 * F.count(F.when(F.col(c).isNull(), 1)) / n_p2).alias(c) for c in p2.columns]).toPandas().iloc[0]
nulos = pd.concat([(100 * p1.isna().mean()).rename("P1"), nulos_p2.rename("P2")], axis=1).round(3)
nulos[(nulos.fillna(0) > 0).any(axis=1)]

,P1,P2
credit_score,0.054,NaN
dti,0.400,NaN
ocltv,0.003,NaN
valuation_method,94.296,NaN
eltv,NaN,10.010
meses_desde_atraso,NaN,88.341
plan_asistencia,NaN,99.484
y_p2,NaN,1.786
y_p2_60,NaN,1.782


Las llaves son únicas en todas las tablas, ninguna tabla de modelado tiene créditos con código 96 ni columnas posteriores al cierre, P1 solo tiene variables de la solicitud y P2 empieza en el primer pago. En P1, valuation_method falta en la mayoría de los créditos de 2019Q1, como en los demás trimestres de 2019 según el perfil de los crudos; si un modelo la usa, el vacío marcaría la cosecha y no el riesgo. En P2, los nulos de meses_desde_atraso son créditos que nunca se han atrasado, los de plan_asistencia son meses sin plan y los de las etiquetas son los últimos meses de los créditos activos. Los meses previos al primer pago se conservan en la capa limpia.

## Pregunta 2: señal de las variables en 2019Q1

In [7]:
carta = p2.where((F.col("dlq_num") == 0) & F.col("y_p2").isNotNull())
atrasos = F.when(F.col("atrasos_12m") == 0, "0").when(F.col("atrasos_12m") == 1, "1") \
    .when(F.col("atrasos_12m") <= 3, "2-3").otherwise("4+")
por_atrasos = carta.groupBy(atrasos.alias("atrasos_12m")).agg(
    F.count("*").alias("creditos_mes"), (100 * F.avg("y_p2")).alias("tasa_pct")).toPandas()
por_atrasos.sort_values("atrasos_12m").round(3)

,atrasos_12m,creditos_mes,tasa_pct
3,0,8450449,0.126
1,1,190545,0.490
0,2-3,103924,0.905
2,4+,130469,1.795


In [8]:
def auc(df, columna, decimales=None):
    valor = F.round(columna, decimales) if decimales is not None else F.col(columna).cast("double")
    t = (df.where(F.col(columna).isNotNull()).groupBy(valor.alias("v"))
         .agg(F.sum("y_p2").alias("pos"), (F.count("*") - F.sum("y_p2")).alias("neg"))
         .toPandas().sort_values("v"))
    negativos_menores = t.neg.cumsum() - t.neg
    return (t.pos * (negativos_menores + 0.5 * t.neg)).sum() / (t.pos.sum() * t.neg.sum())


carta = carta.withColumn("plan_actual", F.col("plan_asistencia").isNotNull())
variables = {"meses_desde_atraso": None, "en_pandemia": None, "eltv": None, "pct_saldo": 4, "atrasos_12m": None,
             "max_atraso_12m": None, "indulgencia_12m": None, "plan_actual": None, "diferimiento": None,
             "diferimiento_mes": None, "saldo_sin_interes": None, "mora_desastre": None, "modificado": None,
             "dif_tasa": 3, "edad": None}
filas = []
for v, d in variables.items():
    a = auc(carta, v, d)
    filas.append({"variable": v, "auc": max(a, 1 - a), "sentido": "+" if a >= 0.5 else "-"})
pd.DataFrame(filas).sort_values("auc", ascending=False).round(3)

,variable,auc,sentido
0,meses_desde_atraso,0.799,-
1,en_pandemia,0.714,+
3,pct_saldo,0.635,+
2,eltv,0.633,+
4,atrasos_12m,0.620,+
5,max_atraso_12m,0.620,+
6,indulgencia_12m,0.554,+
7,plan_actual,0.547,+
10,saldo_sin_interes,0.546,+
8,diferimiento,0.544,+


Con el filtro del primer pago las tasas por atrasos previos casi no cambian frente al EDA. La recencia del último atraso sigue siendo la señal más fuerte entre los créditos que alguna vez se atrasaron, seguida de la proporción de saldo pendiente, el préstamo-valor actual y los atrasos recientes. El diferimiento del mes tiene poca señal por sí solo. La pandemia separa más que casi todas las variables, lo que confirma que distorsiona la etiqueta y que su tratamiento debe decidirse en los experimentos.

## Tablas del tablero de 2019Q1

In [9]:
t_orig = pd.read_parquet(RUTAS["tablero_originacion"])
t_serie = pd.read_parquet(RUTAS["tablero_serie_mensual"])
t_cartera = leer["tablero_cartera_activa"]
serie = t_serie.groupby("period")[["activos", "mora_90"]].sum()
serie["pct_mora_90"] = 100 * serie.mora_90 / serie.activos
pd.Series({
    "créditos en la tabla de originación": t_orig.creditos.sum(),
    "etiquetables de P1 en la tabla de originación": t_orig.p1_etiquetables.sum(),
    "positivos de P1 en la tabla de originación": t_orig.p1_positivos.sum(),
    "activos en la fecha de corte": serie.loc[metricas.loc[TRIMESTRE, "fecha_corte"], "activos"],
    "mes con más mora de 90 días": serie.pct_mora_90.idxmax(),
    "máxima mora de 90 días (% de activos)": round(serie.pct_mora_90.max(), 2),
    "créditos en la cartera activa": t_cartera.count(),
    "créditos de la cartera activa que no están al día": t_cartera.where(F.col("dlq_num") != 0).count(),
}, name="valor", dtype=object).to_frame()

,valor
créditos en la tabla de originación,279861
etiquetables de P1 en la tabla de originación,279060
positivos de P1 en la tabla de originación,680
activos en la fecha de corte,55339
mes con más mora de 90 días,202102
máxima mora de 90 días (% de activos),4.48
créditos en la cartera activa,53605
créditos de la cartera activa que no están al día,0


La tabla de originación conserva todos los créditos y los positivos de P1, la serie mensual reproduce los activos al corte y el pico de mora de 90 días del EDA, y la cartera activa solo tiene créditos al día en la fecha de corte.

## Tablas generadas para 2019Q1

In [10]:
tamanos = pd.DataFrame({
    n: {"filas": df.count(), "columnas": len(df.columns),
        "mb": sum(f.stat().st_size for f in RUTAS[n].glob("*.parquet")) / 1e6}
    for n, df in leer.items()
}).T
spark.stop()
tamanos.astype({"filas": int, "columnas": int}).round(1)

,filas,columnas,mb
originacion,279861,28,5.0
desempeno,9821276,25,127.4
p1,279060,20,4.1
p2,9145339,24,138.2
tablero_originacion,4056,16,0.2
tablero_serie_mensual,4639,15,0.2
tablero_cartera_activa,53605,39,2.0
